In [1]:
import os
import zarr
import numpy as np
from pathlib import Path
import tifffile as tiff
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.interpolate import Rbf
import cv2
import os
import json

DATA_DIR = Path('/root/capsule/data/')

%load_ext autoreload
%autoreload 2

In [3]:
filepaths_dict[key]

PosixPath('/root/capsule/scratch/767022_2025-03-06_coreg_cpsam/755252_2024-12-19_reg-dim-swapped.ome.tif')

In [4]:
subject_id = 767022
save_dir = Path('/root/capsule/scratch/767022_2025-03-06_coreg_cpsam/')
filepaths_json = save_dir / '767022_2025-03-06_filepaths.json'
with open(filepaths_json, 'r') as f:
    filepaths_dict = json.load(f)
for key in filepaths_dict:
    filepaths_dict[key] = Path(filepaths_dict[key])
    assert filepaths_dict[key].exists()
print(filepaths_dict.keys())
# make variables named after each filepath
for key in filepaths_dict:
    globals()[key] = filepaths_dict[key]
hcr_dir = fused_json_file.parent

dict_keys(['czstack_reg_path', 'czstack_reg_dim_swapped_path', 'czstack_seg_path', 'czstack_seg_outline_path', 'czstack_centroid_path', 'hcr_centroid_path', 'fused_json_file'])


In [5]:
landmark_file = save_dir / f'{subject_id}_landmarks.csv'
assert landmark_file.exists(), f'Generate landmarks using BigDataViewer for {subject_id}'

# Find spot data
- Not available for 755252 and 767022: look for /data

In [6]:
spots_488_path = hcr_dir / 'image_spot_detection/channel_488_spots/spots.csv'
if spots_488_path.exists():
    spots_488 = pd.read_csv(spots_488_path)
    spot_488_counts = pd.DataFrame(spots_488['SEG_ID'].value_counts())
    spot_488_counts.rename(columns={'SEG_ID': 'hcr_id'}, inplace=True)
    spot_488_bounding_box_min = spots_488.groupby('SEG_ID').agg({'X':'min','Y':'min','Z':'min'}).rename(columns={'X':'min_x','Y':'min_y','Z':'min_z'})
    spot_488_bounding_box_max = spots_488.groupby('SEG_ID').agg({'X':'max','Y':'max','Z':'max'}).rename(columns={'X':'max_x','Y':'max_y','Z':'max_z'})
    spot_488_bounding_box = pd.concat([spot_488_bounding_box_min, spot_488_bounding_box_max], axis=1)
    spot_488_bounding_box['bb_volume'] = (spot_488_bounding_box['max_x'] - spot_488_bounding_box['min_x'] + 1) * (spot_488_bounding_box['max_y'] - spot_488_bounding_box['min_y'] + 1) * (spot_488_bounding_box['max_z'] - spot_488_bounding_box['min_z'] + 1)
    spot_488_counts = spot_488_counts.merge(spot_488_bounding_box[['bb_volume']], left_index=True, right_index=True)
    spot_488_counts['density'] = spot_488_counts['count'] / spot_488_counts['bb_volume']
    spot_488_counts.set_index('hcr_id', inplace=True)
else:
    spots_path = DATA_DIR / f'cell_data_mean_{subject_id}_R1.csv'
    assert spots_path.exists(), f'No spot_depetection processing nor {spots_path} for {subject_id}'
    spot_488_counts = pd.read_csv(spots_path).query('channel==488')
    spot_488_counts.rename(columns={'cell_id': 'hcr_id', 'mean': 'density'}, inplace=True)
    spot_488_counts.set_index('hcr_id', inplace=True)


## Get HCR centroids and scales

In [7]:
with open(fused_json_file, 'r') as file:
    data = json.load(file)
scale_x = data['dimensions']['x'][0]*4e6
scale_y = data['dimensions']['y'][0]*4e6
scale_z = data['dimensions']['z'][0]*1e6

HCR_cell_centroids = np.load(hcr_centroid_path)

HCR_cell_ids = HCR_cell_centroids[:,3]
HCR_cell_centroids = HCR_cell_centroids[:,:-1]

HCR_cell_centroids_df = pd.DataFrame()
HCR_cell_centroids_df['hcr_cell_id'] = HCR_cell_ids
HCR_cell_centroids_df['hcr_z'] = HCR_cell_centroids[:,0]
HCR_cell_centroids_df['hcr_y'] = HCR_cell_centroids[:,1]
HCR_cell_centroids_df['hcr_x'] = HCR_cell_centroids[:,2]
HCR_cell_centroids_df.set_index('hcr_cell_id', inplace=True)

HCR_cell_centroids_df.head()

,hcr_z,hcr_y,hcr_x
hcr_cell_id,,,
1,172,1835,29
2,173,1770,4
3,173,1835,26
4,173,1773,5
5,174,1834,37


## Get cortical z-stack centroids

In [8]:
czstack_cell_centroids_df = pd.read_csv(czstack_centroid_path, index_col=0)
czstack_cell_centroids_df.head()

,czstack_z,czstack_y,czstack_x
czstack_cell_id,,,
1,39.137353,489.233788,458.070352
2,42.328204,363.942482,313.537841
3,45.054154,438.156623,99.722060
4,46.794484,201.704662,262.273615
5,48.515220,270.163212,165.291775


## apply transformation

In [9]:
czstack_cell_centroids = np.vstack(czstack_cell_centroids_df[['czstack_z','czstack_y','czstack_x']].values)

landmarks = pd.read_csv(landmark_file,header=None)
landmarks = landmarks.loc[landmarks[1]]
landmarks.head()
points_zstack = landmarks.loc[landmarks[1],[4,3,2]].values.astype(np.float32)
points_HCR = landmarks.loc[landmarks[1],[7,6,5]].values.astype(np.float32)

interp_zstacktoHCR_0 = Rbf(points_zstack[:, 0], points_zstack[:, 1], points_zstack[:, 2], points_HCR[:,0],
                           function='thin_plate')
interp_zstacktoHCR_1 = Rbf(points_zstack[:, 0], points_zstack[:, 1], points_zstack[:, 2], points_HCR[:,1],
                           function='thin_plate')
interp_zstacktoHCR_2 = Rbf(points_zstack[:, 0], points_zstack[:, 1], points_zstack[:, 2], points_HCR[:,2],
                           function='thin_plate')

HCR_centroids_est = np.zeros_like(czstack_cell_centroids)

HCR_centroids_est[:,0] = interp_zstacktoHCR_0(czstack_cell_centroids[:, 0],
                                              czstack_cell_centroids[:, 1],
                                              czstack_cell_centroids[:, 2]) / scale_z
HCR_centroids_est[:,1] = interp_zstacktoHCR_1(czstack_cell_centroids[:, 0],
                                              czstack_cell_centroids[:, 1],
                                              czstack_cell_centroids[:, 2]) / scale_y
HCR_centroids_est[:,2] = interp_zstacktoHCR_2(czstack_cell_centroids[:, 0],
                                              czstack_cell_centroids[:, 1],
                                              czstack_cell_centroids[:, 2]) / scale_x


# Estimating based on GFP density in the estimated neighborhood
- Only for pan-inhibitory.
- TODO: dev a protocol for pan-neuronal (when needed)

In [10]:
matched_cells_df = czstack_cell_centroids_df.copy()

# Initialize columns
n_cz = len(matched_cells_df)
matched_cells_df['hcr_cell_id'] = -1
matched_cells_df['hcr_x'] = np.nan
matched_cells_df['hcr_y'] = np.nan
matched_cells_df['hcr_z'] = np.nan
matched_cells_df['distance'] = np.nan

# Merge once, outside the loop
HCR_df = pd.merge(
    HCR_cell_centroids_df.reset_index(),
    spot_488_counts,
    left_index=True,
    right_index=True
)

# Convert to NumPy arrays for speed
HCR_ids = HCR_df.index.to_numpy()                          # (N_hcr,)
HCR_coords = HCR_df[['hcr_z', 'hcr_y', 'hcr_x']].to_numpy()  # (N_hcr, 3)
# HCR_value = HCR_df['count'].to_numpy()                    # (N_hcr,)
HCR_value = HCR_df['density'].to_numpy()                    # (N_hcr,)

N_hcr = len(HCR_ids)

# CZ coordinates that you are matching against
# (assumes HCR_centroids_est[i] is (z, y, x) for CZ cell i)
CZ_coords = np.asarray(HCR_centroids_est)                  # (N_cz, 3)

# Track which HCR cells are already used
used = np.zeros(N_hcr, dtype=bool)

# Storage for assignments (HCR *row positions*, not IDs yet)
assigned_hcr_pos = np.full(n_cz, -1, dtype=int)
assigned_dist = np.full(n_cz, np.nan, dtype=float)

k = 5  # number of nearest neighbors to consider

for i, czsid in tqdm(
    enumerate(matched_cells_df.index.values),
    total=n_cz
):
    # Remaining HCR cells (not yet used)
    remaining = np.flatnonzero(~used)
    if remaining.size == 0:
        # No more HCR cells to assign
        break

    # HCR coords for remaining cells
    rem_coords = HCR_coords[remaining]  # (N_rem, 3)

    # Distances to current CZ cell
    diff = rem_coords - CZ_coords[i]    # (N_rem, 3)
    dist = np.linalg.norm(diff, axis=1) # (N_rem,)

    # Take up to k nearest neighbors
    k_eff = min(k, dist.size)
    # argpartition is faster than full argsort
    nearest_local = np.argpartition(dist, k_eff - 1)[:k_eff]
    nearest_rem_idx = remaining[nearest_local]  # positions in full HCR arrays

    # Among these k neighbors, pick one with max count
    cand_value = HCR_value[nearest_rem_idx]
    best_local = np.argmax(cand_value)
    best_hcr_pos = nearest_rem_idx[best_local]
    best_dist = dist[nearest_local[best_local]]

    # Record assignment
    assigned_hcr_pos[i] = best_hcr_pos
    assigned_dist[i] = best_dist

    # Mark this HCR cell as used
    used[best_hcr_pos] = True

# ------------------------------------------------------------------
# Fill the DataFrame with the results
# ------------------------------------------------------------------

valid = assigned_hcr_pos >= 0  # in case we ran out of HCR cells

matched_cells_df = matched_cells_df.loc[valid].copy()

matched_cells_df['hcr_cell_id'] = HCR_ids[assigned_hcr_pos[valid]]

# Use coordinates from HCR_coords (z, y, x) and scale
matched_cells_df['hcr_z'] = HCR_coords[assigned_hcr_pos[valid], 0] * scale_z
matched_cells_df['hcr_y'] = HCR_coords[assigned_hcr_pos[valid], 1] * scale_y
matched_cells_df['hcr_x'] = HCR_coords[assigned_hcr_pos[valid], 2] * scale_x

matched_cells_df['distance'] = assigned_dist[valid]

# # If you like them sorted by distance (like your original final line):
matched_cells_df = matched_cells_df.sort_values(
    by='distance'
).reset_index()


100%|██████████| 926/926 [00:02<00:00, 313.32it/s]


In [11]:
landmarks = pd.read_csv(landmark_file,header=None)
landmarks = landmarks.loc[landmarks[1]]

landmarks_ext = pd.DataFrame()
# landmarks_ext[0] = ['Pt-'+str(i) for i in range(max_pt+1,max_pt+len(data)+1)]

landmarks_ext[0] = ['cz'+str(int(matched_cells_df['czstack_cell_id'].iloc[i]))+'-hcr'+str(int(matched_cells_df['hcr_cell_id'].iloc[i])) for i in range(len(matched_cells_df))]
# landmarks_ext[0] = ['Pt-'+str(int(i)) for i in range(len(data))]

landmarks_ext[1] = False
# landmarks_ext.loc[matched_cells_df['distance']<5,1] = True
landmarks_ext[2] = matched_cells_df['czstack_x'].values
landmarks_ext[3] = matched_cells_df['czstack_y'].values
landmarks_ext[4] = matched_cells_df['czstack_z'].values
landmarks_ext[5] = matched_cells_df['hcr_x'].values
landmarks_ext[6] = matched_cells_df['hcr_y'].values
landmarks_ext[7] = matched_cells_df['hcr_z'].values

landmarks_ext = pd.concat([landmarks_ext, landmarks])

landmarks_ext

,0,1,2,3,4,5,6,7
0,cz433-hcr29448,False,30.251219,235.065985,241.826180,1361.417154,1119.716282,747.000000
1,cz446-hcr30425,False,181.732078,325.992071,242.953750,1157.204581,966.803486,757.000000
2,cz702-hcr60081,False,379.668766,483.519822,348.863653,890.840355,738.914093,1021.000000
3,cz638-hcr56874,False,260.775387,201.983544,338.232208,1019.089797,1153.258444,986.000000
4,cz666-hcr56429,False,342.939208,284.477380,338.888784,919.449846,1020.076331,982.000000
...,...,...,...,...,...,...,...,...
48,Pt-60,True,118.688245,166.447240,69.914046,1237.341124,1207.693690,316.469752
49,Pt-61,True,45.611392,136.246803,69.408826,1361.118677,1262.652835,318.381374
50,Pt-62,True,391.403932,383.043059,68.359443,892.842708,852.986698,318.967029
51,Pt-63,True,485.528685,291.759349,73.330698,746.699885,969.818390,318.967029


In [12]:
save_fn = save_dir / f'{subject_id}_landmarks_matched_ext_iter1.csv'
# save_fn = save_dir / f'{subject_id}_landmarks_matched_ext_count_iter1.csv'
# save_fn = save_dir / f'{subject_id}_landmarks_matched_ext_density_iter1.csv'
landmarks_ext.to_csv(save_fn, index=False, header=False)

# Misc

## Compare between 'count' and 'density'
- From one example of 755252 - 'Count' is very slightly better (104 vs 94) out of 835

In [ ]:
count_ext = pd.read_csv(save_dir / f'{subject_id}_landmarks_matched_ext_count_iter1.csv', header=None)
density_ext = pd.read_csv(save_dir / f'{subject_id}_landmarks_matched_ext_density_iter1.csv', header=None)

In [33]:
np.sum([name.startswith('cz') for name in count_ext[0].values])

np.int64(835)

In [21]:
count_ext.columns

Index([0, 1, 2, 3, 4, 5, 6, 7], dtype='int64')

In [27]:
count_matched = count_ext[count_ext[1]==True]
np.sum([name.startswith('cz') for name in count_matched[0].values])

np.int64(104)

In [31]:
density_ext

,0,1,2,3,4,5,6,7
0,cz651-hcr65145,True,286.240481,106.109218,326.353707,913.530641,1078.281847,1029.000000
1,cz486-hcr54319,True,24.557377,423.468124,278.927747,1217.383165,644.206813,945.000000
2,cz565-hcr62481,True,106.275271,309.234686,308.454248,1117.743214,797.119609,1004.000000
3,cz51-hcr7065,True,364.156343,200.979387,84.878909,850.392454,1057.564630,502.000000
4,cz59-hcr7340,True,165.486343,479.724098,86.021436,1062.497301,656.045223,507.000000
...,...,...,...,...,...,...,...,...
873,Pt-39,True,167.601601,377.578519,67.431748,1070.254045,783.506449,480.731437
874,Pt-40,True,296.142345,478.155947,63.474857,918.843802,664.436258,437.611367
875,Pt-41,True,445.779037,397.066819,68.210755,732.643504,825.156515,426.831350
876,Pt-42,False,465.665342,265.754557,63.958067,715.493476,983.426769,442.021375


In [32]:
density_matched = density_ext[density_ext[1]==True]
np.sum([name.startswith('cz') for name in density_matched[0].values])

np.int64(94)